# 基于知识图谱的 RAG 系统 —— 演示 Notebook

本 Notebook 复现完整的 **构建图谱 → 图检索 → 生成回答** 流程，方便答辩现场演示。
运行前请确保 `.env` 已配置 DeepSeek Key 与 Neo4j，并已启动 Neo4j。

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

## 1. 构建知识图谱（结构化模式，无需 LLM）

In [ ]:
from config.settings import settings
from src.data.loader import load_input
from src.graph.builder import GraphBuilder
from src.graph.neo4j_client import Neo4jClient

structured, _ = load_input(settings)
entities, relations = [], []
for etype, items in structured.entities.items():
    for item in items:
        entities.append({'name': item['name'], 'type': etype, 'props': item.get('props', {})})
relations = list(structured.relations)

client = Neo4jClient(settings)
builder = GraphBuilder(settings, client)
stats = builder.build(entities, relations, clear_first=True)
print('图谱规模：', stats)

## 2. Graph RAG 问答

In [ ]:
from src.extraction.llm_client import LLMClient
from src.rag.chain import GraphRAGChain

llm = LLMClient(settings)
chain = GraphRAGChain(settings, client, llm)

questions = [
    '克里斯托弗·诺兰导演了哪些电影？',
    '莱昂纳多·迪卡普里奥参演过哪些电影？',
    '肖申克的救赎是什么类型的电影？',
]
for q in questions:
    r = chain.answer(q, show_context=True)
    print('\n【问题】', r['question'])
    print('【答案】', r['answer'])
    print('【命中实体】', r['entities'])

## 3. 关闭连接

In [ ]:
client.close()

> 提示：可在浏览器打开 `http://localhost:7474`，用 `MATCH (n:Entity) RETURN n LIMIT 25` 查看图谱可视化。